<a href="https://colab.research.google.com/github/AlexeyTri/SemMed_fall25/blob/main/HW/HW_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Задание**
Цель работы в написании классификатора, для выборки sklearn.datasets.fetch_covtype()

Задачи:
1. Загрузить выборку -> сформировать три датасета: train, valid, test (4 балла)
2. Построить модель, функцию обучения train, функцию проверки качества работы модели evaluate -> обучить модель, замерить метрики качества, выйти на минимально необходимый уровень 93% на test (то есть после обучения модели, вы подаете тестовые данные и проверяете качество предсказания) (4 балла)
3. Определить оптимальные параметры модели при помози optuna (2 балла)

In [ ]:
!pip install optuna
!pip install torchmetrics
import optuna
import torch
import numpy as np
import sklearn
import torchmetrics
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
import torch.nn as nn
from sklearn.datasets import fetch_covtype

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
elif torch.cpu.is_available():
    device = 'cpu'

device

'cpu'

In [ ]:
torch.manual_seed(42)

**1. Загрузить выборку -> сформировать три датасета: train, valid, test (4 балла)**

In [ ]:
# загрузите данные fetch_covtype()
# обратите внимание, что это функция, которая имеет ряд параметров, может чтото из них стоит применить?

X, y = # your code here

In [ ]:
# при работе с классификаторами, стоит обратить внимание на индексацию классов. Если в выборке классы индексируют как придется, то при подаче в модель, они должны идти с 0 до n_classes
# выполните предобратобку y

y = # your code here

assert sum(y == 4) == 9493

In [ ]:
# сформируйте три выборки train, valid, test

X_train_full, X_test, y_train_full, y_test = # your code here

X_train, X_valid, y_train, y_valid = # your code here


In [ ]:
# выполните стандартизацию данных

# your code here


In [ ]:
# сформируйте три DataLoader: train_loader, valid_loader, test_loader
# размер batch = 32
# ВНИМАНИЕ, целевые переменные должны иметь тип данных LongTensor

# your code here

train_loader = # your code here
valid_loader = # your code here
test_loader = # your code here


In [ ]:
# заполните пропуски в функции train

def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = # your code here
            y_pred = # your code here
            loss = # your code here
            total_loss += loss.item()
            # your code here
            # your code here
            # your code here
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [ ]:
# заполните пропуски в функции evaluate

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = # your code here
            y_pred = # your code here
            metric.update(y_pred, y_batch)
    return metric.compute()

In [ ]:
# заполните пропуски в классе NewClass, на выходе должен получиться трехслойный линейный классификатор

class NewClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(

            # your code here
        )

    def forward(self, X):
        return self.mlp(X)

torch.manual_seed(42)

model = NewClassifier(# your code here).to(device)
xentropy = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
_ = train2(model, optimizer, xentropy, accuracy, train_loader, valid_loader,
           n_epochs=20)

In [ ]:
# заполните пропуски в функции objective

def objective(trial):
    learning_rate = trial.suggest_float(# your code here)
    n_hidden = trial.suggest_int(# your code here)
    model = NewClassifier(# your code here).to(device)
    optimizer = torch.optim.SGD(# your code here)
    xentropy = # your code here
    accuracy = # your code here
    history = # your code here
    validation_accuracy = max(history["valid_metrics"])
    return validation_accuracy

In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

In [ ]:
study.best_params